# 02 — Feature engineering: Elo, form, strength, head-to-head

Every feature in this notebook is computed from matches **before** the one it
describes. The tests in `tests/test_no_leakage.py` prove that mechanically; this
notebook is the human check — do the numbers look like football?

Three things to look for:

1. Elo trajectories should match the sport's history (Brazil and Germany high
   and stable, a nation's curve bending at a golden generation).
2. `elo_diff` should be clearly related to the points a team takes, but far from
   deterministic — if it explained everything, the problem would be trivial.
3. `attack` and `defence` should spread around 1.0, since they are ratios to the
   league average.

In [1]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src import config, data, features

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 130)
FIG = config.FIGURES_DIR

## 1. Build the feature table

One row per team per match, all values pre-match. `history_matches` counts how
many matches the team had played *before* this one — it is the "how much should
you trust this row's form numbers?" column, and it is why filling missing form
with the league average is harmless.

In [2]:
if not config.RESULTS_CSV.exists():
    print("!! using SYNTHETIC data (drop the Kaggle results.csv in data/raw/ for real numbers)")
    data.generate_synthetic_results(config.RESULTS_CSV)

matches = data.load_results()
feats = features.build_feature_table(matches)

print(f"matches: {len(matches):,}   team-rows: {len(feats):,}")
print(f"features: {len(features.FEATURE_COLUMNS)}")
feats[features.FEATURE_COLUMNS].describe().T[["mean", "std", "min", "max"]].round(3)

matches: 5,999   team-rows: 11,998
features: 19


,mean,std,min,max
elo_team,1501.171,124.013,1091.622,1907.083
elo_opponent,1501.171,124.013,1091.622,1907.083
elo_diff,0.000,176.341,-711.459,711.459
home_advantage,0.390,0.488,0.000,1.000
history_matches,125.274,73.519,0.000,300.000
form_gf_5,1.696,0.786,0.000,6.600
form_ga_5,1.693,0.743,0.000,6.200
form_points_5,1.388,0.674,0.000,3.000
form_win_5,0.391,0.240,0.000,1.000
form_gf_10,1.695,0.660,0.000,6.000


## 2. Elo trajectories

Ratings are read *before* each match and updated after it, so a team's curve
moves only when it plays. Below are the six strongest sides by current rating.

In [3]:
elo = features.add_elo_ratings(matches)
long_elo = pd.concat([
    elo[["date", "home_team", "elo_home_post"]]
    .rename(columns={"home_team": "team", "elo_home_post": "elo"}),
    elo[["date", "away_team", "elo_away_post"]]
    .rename(columns={"away_team": "team", "elo_away_post": "elo"}),
]).sort_values("date")

latest = long_elo.groupby("team").tail(1).sort_values("elo", ascending=False)
top_teams = latest.head(6)["team"].tolist()

fig, ax = plt.subplots(figsize=(9, 4.5))
for team in top_teams:
    series = long_elo[long_elo["team"] == team]
    ax.plot(series["date"], series["elo"], linewidth=1.2, label=team)
ax.set_ylabel("Elo rating")
ax.set_title("Elo trajectories of the strongest current teams")
ax.legend(ncol=3, fontsize=9)
fig.tight_layout()
fig.savefig(FIG / "05_elo_trajectories.png", dpi=130)
plt.close(fig)

latest.head(12).assign(date=lambda d: d["date"].dt.date).reset_index(drop=True)

,date,team,elo
0,2023-12-25,Uruguay,1837.652315
1,2023-11-28,Wales,1803.188717
2,2023-12-25,Costa Rica,1759.780884
3,2023-12-11,Denmark,1732.913389
4,2023-12-01,Germany,1689.217612
5,2023-12-23,Egypt,1680.950434
6,2023-12-31,USA,1675.917021
7,2023-10-25,Czech Republic,1667.913217
8,2023-11-11,South Korea,1653.186173
9,2023-05-22,Canada,1645.721079


In [4]:
fig, ax = plt.subplots(figsize=(7, 4.5))
top15 = latest.head(15).iloc[::-1]
ax.barh(top15["team"], top15["elo"], color="steelblue")
ax.set_xlabel("Elo rating")
ax.set_title("Current Elo ratings (top 15)")
ax.axvline(features.START_RATING, color="grey", linestyle="--", linewidth=1)
fig.tight_layout()
fig.savefig(FIG / "06_current_elo.png", dpi=130)
plt.close(fig)
print("saved 06_current_elo.png")

saved 06_current_elo.png


## 3. Does the rating actually predict results?

Bucket `elo_diff` and look at the average points a team takes. A monotone rise is
the minimum bar for a usable rating. The slope *should* flatten at the extremes:
a 400-point gap is not four times as informative as a 100-point one.

In [5]:
feats["points"] = np.where(feats["result"] == "W", 3, np.where(feats["result"] == "D", 1, 0))
bins = [-1200, -300, -150, -50, 50, 150, 300, 1200]
feats["elo_bucket"] = pd.cut(feats["elo_diff"], bins)

by_bucket = (feats.groupby("elo_bucket", observed=True)
             .agg(mean_points=("points", "mean"), n=("points", "size")))
print(f"corr(elo_diff, points) = {np.corrcoef(feats['elo_diff'], feats['points'])[0, 1]:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
labels = [f"{int(iv.left)}..{int(iv.right)}" for iv in by_bucket.index]
ax.bar(labels, by_bucket["mean_points"], color="steelblue")
for i, (pts, n) in enumerate(zip(by_bucket["mean_points"], by_bucket["n"])):
    ax.text(i, pts, f"n={n}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Elo difference (team minus opponent)")
ax.set_ylabel("mean points per match (0-3)")
ax.set_title("Points taken by Elo difference bucket")
ax.axhline(1.0, color="grey", linestyle="--", linewidth=1)
fig.tight_layout()
fig.savefig(FIG / "07_elo_vs_points.png", dpi=130)
plt.close(fig)
by_bucket.round(2)

corr(elo_diff, points) = 0.284


,mean_points,n
elo_bucket,,
"(-1200, -300]",0.62,545
"(-300, -150]",0.96,1763
"(-150, -50]",1.18,2294
"(-50, 50]",1.38,2794
"(50, 150]",1.60,2294
"(150, 300]",1.84,1763
"(300, 1200]",2.22,545


## 4. Feature correlations

Read this for two things: redundancy (features that duplicate each other make
coefficients unstable in a GLM) and direction (attack should correlate
positively with goals scored, defence negatively, since a high `defence` ratio
means *conceding* a lot).

In [6]:
matrix = feats[features.FEATURE_COLUMNS + ["goals_for", "goals_against"]].corr()

fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(matrix, cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.7}, linewidths=0.3)
ax.set_title("Feature correlation matrix")
fig.tight_layout()
fig.savefig(FIG / "08_feature_correlations.png", dpi=130)
plt.close(fig)

matrix[["goals_for", "goals_against"]].drop(index=["goals_for", "goals_against"]).round(2)

,goals_for,goals_against
elo_team,0.22,-0.14
elo_opponent,-0.14,0.22
elo_diff,0.25,-0.25
home_advantage,0.15,-0.10
neutral,-0.06,-0.06
history_matches,0.00,0.00
form_gf_5,0.20,-0.00
form_ga_5,-0.01,0.13
form_points_5,0.13,-0.08
form_win_5,0.13,-0.06


## Takeaways

* **Elo behaves like a football rating**: slow-moving, mean-reverting, and it
  separates the top nations without exploding at the extremes.
* **`elo_diff` is a strong but imperfect signal** — mean points rise steadily
  across buckets and flatten at the top, which is what a well-scaled rating
  looks like. If you ever see a perfect relationship here, you have a leak.
* **`attack` and `defence` centre on 1.0** because they are ratios to the
  time-varying league average. That makes their coefficients interpretable in
  the Poisson model: `attack` multiplies the expected goal rate.
* **`elo_team`, `elo_opponent` and `elo_diff` are collinear** by construction.
  The models will handle that differently (a GLM should drop one, trees do not
  care), which is exactly the kind of thing step 4-5 needs to compare.

Next (step 3): time-based splits, the metric library (MAE, RMSE, Poisson
deviance, log loss, RPS with bootstrap intervals), and the baselines every model
has to beat.